<a href="https://colab.research.google.com/github/TomonoriGH/colab_git/blob/main/arbit_nn.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ニューラルネットワークに3桁*3桁の掛け算をさせたかった(2026/09/15)

## 結論
できなかった。

## やったこと
1. ColabのGeminiに「シンプルなNN作成+学習+評価のコードを作ってとお願い」  
（コミットないですごめんなさいm(__)m）  
「example」の下のセルはGeminiの生成物  
「selfmade」の下のセルはそれをもとに自分で書いたものです。
2. 学習データとしてかける数、かけられる数(X_train)、答え(y_train) を作成  
  例 : 入力 [9,8,7,6,5,4] → 出力 [6,4,5,4,9,8]　　(987×654 = 645498 だから)
3. 学習・評価

## 気づきなど
 - 6桁の出力の最上位は大体誤差が1以内、他は3.0～4.9くらい。[(このセルを実行すると評価が見れます)](#scrollTo=JktjrtUIhLU_)
 - 誤差関数を(Geminiが書いた)CrossEntropyからMSELossに変えたが、効果なし。
 - Gemini生成のコード中の変数「loss」の意味がわからなかった。
 - 多分LinearとReLUだけでは構造的に掛け算ができないのだと思う。
&nbsp;  
&nbsp;  
&nbsp;  
&nbsp;  
&nbsp;  
## 以下コード

### 1. 設定とモデルの定義
入力層と出力層のサイズを指定し、シンプルなニューラルネットワークを定義します。

In [1]:
!pip install torchinfo

#### example

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np

# --- パラメータ設定 ---
INPUT_SIZE = 10   # 入力層の大きさ
OUTPUT_SIZE = 2   # 出力層の大きさ (例: 2クラス分類)
HIDDEN_SIZE = 64  # 隠れ層の大きさ
BATCH_SIZE = 16
LEARNING_RATE = 0.01
EPOCHS = 20

# モデルの定義
class SimpleNet(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super(SimpleNet, self).__init__()
        self.fc1 = nn.Linear(input_size, hidden_size)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        return x

model = SimpleNet(INPUT_SIZE, HIDDEN_SIZE, OUTPUT_SIZE)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
print("Model initialized.")

Model initialized.


#### selfmade

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
from torchinfo import summary

# --- パラメータ設定 ---
INPUT_SIZE = 6   # 入力層の大きさ
OUTPUT_SIZE = 6   # 出力層の大きさ (例: 2クラス分類)
NUM_SAMPLE = 10000
HIDDEN_SIZE = 64  # 隠れ層の大きさ
HIDDEN_SIZE = 6  # 隠れ層の大きさ
BATCH_SIZE = 16
LEARNING_RATE = 0.01
EPOCHS = 200
LAYER_LENGTH = 20

model = nn.Sequential(*[
        nn.Linear(
              INPUT_SIZE if i == 0 else HIDDEN_SIZE,
              OUTPUT_SIZE if i == LAYER_LENGTH - 1 else HIDDEN_SIZE
        )
      if ii == 0 else
      nn.ReLU()
      for i in range(LAYER_LENGTH)
      for ii in range(1 if i == LAYER_LENGTH - 1 else 2)
    ])
criterion = nn.CrossEntropyLoss()
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
print("Model initialized.")
summary(model)

Model initialized.


Layer (type:depth-idx)                   Param #
Sequential                               --
├─Linear: 1-1                            42
├─ReLU: 1-2                              --
├─Linear: 1-3                            42
├─ReLU: 1-4                              --
├─Linear: 1-5                            42
├─ReLU: 1-6                              --
├─Linear: 1-7                            42
├─ReLU: 1-8                              --
├─Linear: 1-9                            42
├─ReLU: 1-10                             --
├─Linear: 1-11                           42
├─ReLU: 1-12                             --
├─Linear: 1-13                           42
├─ReLU: 1-14                             --
├─Linear: 1-15                           42
├─ReLU: 1-16                             --
├─Linear: 1-17                           42
├─ReLU: 1-18                             --
├─Linear: 1-19                           42
├─ReLU: 1-20                             --
├─Linear: 1-21             

### 2. ミニバッチ生成関数とダミーデータ
学習に使用するサンプルデータと、バッチを取得する関数を用意します。

#### example

In [ ]:
# ダミーデータの作成 (入力: 100サンプル, 正解ラベル: 0 or 1)
X_train = torch.randn(100, INPUT_SIZE)
y_train = torch.randint(0, OUTPUT_SIZE, (100,))

def get_mini_batches(X, y, batch_size):
    indices = np.arange(len(X))
    np.random.shuffle(indices)
    for i in range(0, len(X), batch_size):
        batch_idx = indices[i:i + batch_size]
        yield X[batch_idx], y[batch_idx]

print("Data and batch generator ready.")

#### selfmade

In [3]:
def ylinside(l) :
  l = [str(e) for e in l]
  v1 = int("".join(l[:3])) * int("".join(l[3:]))
  return [int(e) for e in list(str(v1).zfill(6))]

xl,yl,X_train,y_train = [None] * 4

def gen_train_data() :
  global xl,yl,X_train,y_train
  xl = np.random.randint(0,10,size=(NUM_SAMPLE,INPUT_SIZE)).tolist()
  yl = [ylinside(xl[i]) for i in range(NUM_SAMPLE)]

  X_train = torch.tensor(xl,dtype=torch.float)
  y_train = torch.tensor(yl,dtype=torch.float)

def get_mini_batches(X, y, batch_size):
    gen_train_data()
    indices = np.arange(len(X))
    np.random.shuffle(indices)
    for i in range(0, len(X), batch_size):
        batch_idx = indices[i:i + batch_size]
        yield X[batch_idx], y[batch_idx]


print("Data and batch generator ready.")

Data and batch generator ready.


### 3. 学習実行 (ボタン一つで実行相当)
このセルを実行することで学習が始まります。

In [5]:
def train():
    model.train()
    for epoch in range(EPOCHS):
        total_loss = 0
        for batch_X, batch_y in get_mini_batches(X_train, y_train, BATCH_SIZE):
            optimizer.zero_grad()
            outputs = model(batch_X)
            loss = criterion(outputs, batch_y)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        if (epoch + 1) % 5 == 0:
            print(f"Epoch [{epoch+1}/{EPOCHS}], Loss: {total_loss:.4f}")
    print("Training completed.")

train()

Epoch [5/200], Loss: 4880.5887
Epoch [10/200], Loss: 4876.0305
Epoch [15/200], Loss: 4900.6652
Epoch [20/200], Loss: 4883.8770
Epoch [25/200], Loss: 4874.0520
Epoch [30/200], Loss: 4891.2284
Epoch [35/200], Loss: 4901.2493
Epoch [40/200], Loss: 4881.6349
Epoch [45/200], Loss: 4919.8952
Epoch [50/200], Loss: 4915.7974
Epoch [55/200], Loss: 4901.8944
Epoch [60/200], Loss: 4881.8597
Epoch [65/200], Loss: 4865.9052
Epoch [70/200], Loss: 4860.2645
Epoch [75/200], Loss: 4881.8009
Epoch [80/200], Loss: 4891.5834
Epoch [85/200], Loss: 4872.2426
Epoch [90/200], Loss: 4894.2136
Epoch [95/200], Loss: 4905.9988
Epoch [100/200], Loss: 4895.7875
Epoch [105/200], Loss: 4895.5740
Epoch [110/200], Loss: 4878.5283
Epoch [115/200], Loss: 4913.2093
Epoch [120/200], Loss: 4924.4629
Epoch [125/200], Loss: 4859.9198
Epoch [130/200], Loss: 4868.1622
Epoch [135/200], Loss: 4923.7277
Epoch [140/200], Loss: 4886.7586
Epoch [145/200], Loss: 4875.2764
Epoch [150/200], Loss: 4871.4507
Epoch [155/200], Loss: 4891.67

### 4. 使用・評価 (ボタン一つで実行相当)
学習済みモデルを使用して推論を行う例です。

#### example

In [12]:
def evaluate():
    model.eval()
    # 新しい未知のデータを作成
    test_input = torch.randn(5, INPUT_SIZE)

    with torch.no_grad():
        predictions = model(test_input)
        # 最大値のインデックスを取得 (クラス予測)
        _, predicted_classes = torch.max(predictions, 1)

    print("Input Data Shape:", test_input.shape)
    print("Predicted Classes:", predicted_classes.numpy())

evaluate()

Input Data Shape: torch.Size([5, 6])
Predicted Classes: [0 0 0 0 0]


#### selfmade
<div id="JktjrtUIhLU_"></div>

In [6]:
def evaluate():
    model.eval()
    # 新しい未知のデータを作成
    test_input,answer = next(get_mini_batches(X_train,y_train,1))
    # -- 任意の数値で試したい場合はここを使おう（それぞれ999以下で）--
    # factors = [123,456]
    # answer_ = factors[0] * factors[1]
    # test_input = torch.tensor([[int((f / 10 ** (2-i) % 10)) for f in factors for i in range(3)]],dtype = torch.float)
    # answer = torch.tensor([[int((answer_ / 10 ** (5-i) % 10))  for i in range(6)]],dtype = torch.float)
    # -------
    print("input",test_input[0])
    print("answer",answer[0])

    with torch.no_grad():
        predictions = model(test_input)
        print(predictions[0])

evaluate()

input tensor([4., 0., 2., 9., 6., 1.])
answer tensor([3., 8., 6., 3., 2., 2.])
tensor([2.0183, 4.0639, 4.3591, 4.5625, 4.4555, 3.6234])
